# 2026-11-08 Fair matching-speed comparison (8 bins)

**Fixed:** regenerated 2026-10-14 descriptor cache, original 8-bin quantile edges, exact MNN (smallest index on ties), 2 threads, 3 warmups + 9 repeats per pair, rotating method order. XFeat extraction is excluded.

**Changed:** matching representation / distance / implementation:
`official_torch_cosine` (XFeat `match`, two GEMMs), `float_blas_single_gemm`, `float_cpp_fused`, `thermo8_L1_original` (unchanged 2026-10-14 kernel), `byte8_L1`, `byte8_L2`, `int8_dot` (per-tensor symmetric int8).

Diagnostic run: no setting is selected from these results.

**Before running:** upload to Google Drive
1. this whole folder → `MyDrive/Colab Notebooks/2026-11-08_fair_speed/`
2. `CSE527/data/2026-10-14_regen_cache/` → `MyDrive/Colab Notebooks/2026-10-14_regen_cache/`

Runtime: CPU only (Runtime → Change runtime type → CPU). Expected run time roughly 10–20 min (not measured on Colab).

In [ ]:
import os
os.environ['OMP_NUM_THREADS']='2'; os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['MKL_NUM_THREADS']='2'
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/Colab Notebooks')
EXP=BASE/'2026-11-08_fair_speed'
CACHE=BASE/'2026-10-14_regen_cache'
for p in [EXP/'run_speed.py',EXP/'speed_kernels.py',EXP/'inputs'/'fixed_quantile_edges.npz',CACHE/'coco'/'calibration.npy',CACHE/'hpatches_cache'/'hpatches_pairs.npz']:
    assert p.exists(),p
print('inputs found')

In [ ]:
import subprocess,sys,importlib.util
for module,package in {'cv2':'opencv-python-headless','pandas':'pandas','torch':'torch'}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
# Copy to local disk so Drive I/O does not enter the timing loop.
subprocess.run(['rm','-rf','/content/run','/content/cache'],check=True)
subprocess.run(['cp','-r',str(EXP),'/content/run'],check=True)
subprocess.run(['cp','-r',str(CACHE),'/content/cache'],check=True)
print(subprocess.run('lscpu | grep -E "Model name|^CPU\\(s\\)|Flags" | cut -c1-200',shell=True,capture_output=True,text=True).stdout)

In [ ]:
import time
start=time.time()
r=subprocess.run([sys.executable,'/content/run/run_speed.py','/content/cache','/content/run/inputs/fixed_quantile_edges.npz',
                  '/content/run/inputs/orig_code','/content/run/results_colab'],capture_output=True,text=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:]); print('minutes',(time.time()-start)/60)
assert r.returncode==0

In [ ]:
# Save results back to Drive (new folder; nothing existing is overwritten)
out=EXP/f'results_colab_{time.strftime("%Y%m%d_%H%M%S")}'
subprocess.run(['cp','-r','/content/run/results_colab',str(out)],check=True)
print('saved to',out)